# Estudio de tokens sobre bd CORPES
CORPES es un CORPus de ESpañol mantenido por la RAE

In [ ]:
from IPython.display import clear_output
import gc
import psutil
print(gc.collect())
print("Memoria:", psutil.virtual_memory())

In [2]:
import importlib
import dfrtokenuniverse.splitter
import dfrtokenuniverse.word_inventory
import dfrtokenuniverse.constantes

In [ ]:
from dfrtokenuniverse.splitter import SplitTextTokens
from dfrtokenuniverse.word_inventory import WordInventory
from dfrtokenuniverse.constantes import KDfrNlp
K = KDfrNlp()
splitter = SplitTextTokens()
invent = WordInventory()
splitter.version

In [4]:
camino = 'D:\\datos\\'

In [ ]:
cont = 0
with open(camino + "listas_dp_lemas.tsv", 'r', encoding="utf-8") as f:
  for linea in f.readlines():
    if cont < 10:
      print(linea)
    if "yohagoloque" in linea.lower():
      print("ENCONTRADO:", linea)
      break
    cont += 1

In [ ]:
import pandas as pd
df_lemas = pd.read_csv(
    camino + "listas_dp_lemas.tsv",
    delimiter="\t", index_col=0, skiprows=4,
    #names=['Lema', 'Clase', 'Frecuencia', 'Frec.norm.1', 'Frec.norm.2']
    encoding="utf-8"
).dropna()
df_lemas.head(5)

In [19]:
class ProgressText():
    def __init__(self, anchura=150):
        self.anchura = anchura
    def init(self, total):
        self.total = total
        self.actual = 0
        self.incremento = (self.total // self.anchura if self.total > self.anchura else 1)
    def header(self):
        print("".join(["|"]*self.anchura))
    def count(self):
        self.actual += 1
        if self.actual % self.incremento == 0:
            print(".", end="")
        if self.actual >= self.total:
            print("!")
progreso = ProgressText()
    

In [ ]:
dic_tokengramas = {}
dic_silabas = {}
progreso.init(len(df_lemas))
progreso.header()
for row in df_lemas.iterrows():
    lema = row[1]['Lema'].lower()
    apariciones = int(row[1]['Frecuencia'])
    for tokengrama in splitter.text_tokens(lema, dic_silabas):
        if tokengrama in dic_tokengramas:
            dic_tokengramas[tokengrama] += apariciones
        else:
            dic_tokengramas[tokengrama] = apariciones
    progreso.count()
print()
print(f"Tenemos {len(dic_tokengramas)} tokengramas")

In [ ]:
cont_tokengramas_by_len = {}
acum_tokengramas_by_len = {}
cont_codes = 0
progreso.init(len(dic_tokengramas))
progreso.header()
for tokengrama in dic_tokengramas:
    if isinstance(tokengrama, str):
        longitud = len(tokengrama)
        if longitud in cont_tokengramas_by_len:
            cont_tokengramas_by_len[longitud] += 1
            acum_tokengramas_by_len[longitud] += dic_tokengramas[tokengrama]
        else:
            cont_tokengramas_by_len[longitud] = 1
            acum_tokengramas_by_len[longitud] = dic_tokengramas[tokengrama]
    else:
        cont_codes += 1
    progreso.count()
print()
print(f"Tenemos {len(cont_tokengramas_by_len)} longitudes de tokengramas")
print(f"Tenemos {cont_codes} códigos")
for longitud in sorted(cont_tokengramas_by_len.keys()):
    print(f"con {longitud}-gramas: {cont_tokengramas_by_len[longitud]} ({acum_tokengramas_by_len[longitud]} apariciones {int(round(acum_tokengramas_by_len[longitud] / cont_tokengramas_by_len[longitud]))} por tokengrama)")

In [ ]:
del df_lemas
print(gc.collect())
print("Memoria:", psutil.virtual_memory())

In [ ]:
for longitud in sorted(cont_tokengramas_by_len.keys()):
    cont = 0
    for tokengrama in dic_tokengramas:
        if isinstance(tokengrama, str):
            if longitud == len(tokengrama):
                print(tokengrama, end=", ")
                cont += 1
                if cont > 10:
                    break
    print()

In [ ]:
dic_silabas_len = {}
dic_silabas = {}
dic_splitter = {}
print("".join(["!"]*150))
cada = len(dic_ngramas[1]) // 150
cont = 0
for lema in dic_ngramas[1]:
    try:
        silabas = auxplitter.silabas(lema, dic_splitter)
    except Exception as e:
        print(f"Error en {lema}")
        raise f"error: {e}"
    lensil = len(silabas)
    if lensil in dic_silabas_len:
        dic_silabas_len[lensil][lema] = silabas.copy() 
    else:
        dic_silabas_len[lensil] = {lema: silabas.copy()}       
    for silaba in silabas:
        if silaba in dic_silabas:
            dic_silabas[silaba] += dic_ngramas[1][lema]
        else:
            dic_silabas[silaba] = dic_ngramas[1][lema]
    cont += 1
    if cont % cada == 0:
        print(".", end="")
print()
print(f"Tenemos {len(dic_silabas)} silabas")
print(f"Hemos usado {len(dic_splitter)} troceadores")

In [ ]:
cont = 0
silabicos_efectivos = "# Divisores de silaba optimizados:\n"
for silabico in  sorted(dic_splitter, key=lambda x: str(dic_splitter[x]).zfill(11) + x, reverse=True):
    silabicos_efectivos += f"'{silabico}': {K.SILABICOS[silabico]}, ".rjust(7)
    cont +=1
    if cont % int(100/8) == 0:
        silabicos_efectivos += "\n"
silabicos_efectivos += "\n# Divisories de silaba teoricos (Sin optimizar o que no han aparecido aun en las pruebas):\n"
for silabico in K.SILABICOS:
    if silabico not in silabicos_efectivos:
        silabicos_efectivos += f"'{silabico}': {K.SILABICOS[silabico]}, ".rjust(7)
        cont +=1
        if cont % int(100/8) == 0:
            silabicos_efectivos += "\n"
print(silabicos_efectivos)

In [ ]:
for lensil in sorted(dic_silabas_len):
    print(
        f"Para longitud {lensil} tenemos {len(dic_silabas_len[lensil])} {lensil}-gramas" +
        f" sample: ", end= ""
    )
    for lema in [x for x in dic_silabas_len[lensil]][:20]:
        print(f"{lema}({dic_silabas_len[lensil][lema]}), ", end="")
    print()

In [ ]:
for lensil in sorted(dic_silabas_len):
    for lema in dic_silabas_len[lensil]:
        ko = False
        for silaba in dic_silabas_len[lensil][lema]:
            if len(silaba) > 4 or len(silaba) == 1 and silaba not in "aeiouáéíóú":
                ko = True
        if ko:
            print(f"{lema}: {dic_silabas_len[lensil][lema]}")
                

In [16]:
auxplitter.K.SILABICOS = {
    # Divisores de silaba optimizados:
    'nt': 1, '*do': 0, '*co': 0, '*ci': 0, '*na': 0, '*ra': 0, '*li': 0, '*ri': 0, '*me': 0, '*to': 0, '*ca': 0, '*ni': 0, 
    '*no': 0, '*ta': 0, '*mi': 0, '*la': 0, '*da': 0, '*ti': 0, '*ro': 0, 'nc': 1, '*si': 0, '*rr': 0, '*sta': 1, '*za': 0, 
    '*te': 0, 'nd': 1, '*le': 0, '*ma': 0, '*lo': 0, '*di': 0, '*pe': 0, 'ea': 1, 'sc': 1, '*de': 0, '*ll': 0, 'eo': 1, 
    'sm': 1, '*sa': 0, 'ct': 1, '*ne': 0, 'rt': 1, 'mp': 1, '*so': 0, '*bl': 0, 'ía': 1, '*fi': 0, '*cr': 0, '*tr': 0, 
    '*ga': 0, '*re': 0, '*mo': 0, 'sp': 1, '*pa': 0, 'rc': 1, '*ce': 0, 'rm': 1, '*ch': 0, 'ntr': 1, '*vi': 0, 'ng': 1, 
    '*se': 0, '*po': 0, '*cu': 0, '*gr': 0, '*bi': 0, '*qu': 0, '*vo': 0, 'lt': 1, 'mb': 1, '*br': 0, '*sti': 1, '*va': 0, 
    '*pi': 0, '*gi': 0, '*pr': 0, 'str': 1, 'rd': 1, '*ba': 0, 'nf': 1, '*go': 0, '*be': 0, '*ve': 0, '*gu': 0, '*tu': 0, 
    '*ste': 1, 'rg': 1, 'oa': 1, 'oe': 1, '*rí': 0, 'rn': 1, 'lm': 1, '*zo': 0, '*ge': 0, '*je': 0, '*ja': 0, '*fo': 0, 
    '*du': 0, '*dr': 0, '*bo': 0, 'rb': 1, '*lu': 0, '*fe': 0, '*xi': 0, 'cc': 1, '*ló': 0, '*pl': 0, '*sto': 1, '*fa': 0, 
    '*ño': 0, '*cl': 0, 'rp': 1, '*nse': 1, 'pt': 1, 'ee': 1, 'nv': 1, '*mu': 0, '*su': 0, '*gí': 0, '*pu': 0, '*nu': 0, 
    '*bu': 0, '*jo': 0, '*nsi': 1, '*ña': 0, '*fr': 0, '*lí': 0, 'rv': 1, 'sf': 1, 'rf': 1, 'nm': 1, 'xp': 1, 'lc': 1, 
    'xtr': 1, 'ltr': 1, 'oo': 1, '*rse': 1, 'gn': 1, '*nsa': 1, '*tó': 0, 'nst': 2, 'ae': 1, 'nz': 1, '*ní': 0, '*má': 0, 
    'ld': 1, '*fí': 0, 'sq': 1, 'rq': 1, '*ya': 0, '*ru': 0, 'nq': 1, 'cn': 1, 'sl': 1, 'rl': 1, '*fl': 0, '*nsu': 1, 
    '*fu': 0, 'xc': 1, '*nso': 1, 'lg': 1, 'lv': 1, '*rsi': 1, 'scr': 1, 'xt': 1, '*né': 0, 'ls': 1, 'nj': 1, '*ró': 0, 
    '*nó': 0, 'sg': 1, '*gl': 0, '*ju': 0, '*ji': 0, '*xa': 0, 'lp': 1, '*rso': 1, '*mé': 0, '*tí': 0, '*mí': 0, '*stu': 1, 
    '*gé': 0, '*có': 0, 'aa': 1, '*yo': 0, '*ps': 0, '*xo': 0, 'lf': 1, '*hi': 0, 'tn': 1, 'iner': 2, '*ye': 0, 'sv': 1, 
    '*té': 0, '*rá': 0, 'ao': 1, '*tá': 0, '*ná': 0, '*lé': 0, '*hu': 0, '*sí': 0, '*rsa': 1, 'nr': 1, 'sb': 1, 'lb': 1, 
    'bc': 1, '*mó': 0, 'eí': 1, 'ncr': 1, 'gm': 1, 'antia': 4, 'sn': 1, 'ío': 1, '*cá': 0, '*lá': 0, 'pc': 1, '*mn': 1, 
    'ii': 1, 'nn': 1, 'antie': 4, '*cé': 0, '*cí': 0, 'rz': 1, 'ioe': 2, '*xe': 0, '*pí': 0, '*gó': 0, 'sd': 1, 'aí': 1, 
    '*ñi': 0, 'nh': 1, 'ck': 1, '*ñe': 0, '*pó': 0, '*fó': 0, '*xu': 0, '*zu': 0, 'bd': 1, '*só': 0, '*dí': 0, 'nl': 1, 
    'sr': 1, 'bst': 2, 'bj': 1, 'stc': 2, 'ohe': 1, 'antii': 4, '*yu': 0, '*dó': 0, '*pá': 0, 'ioi': 2, '*zó': 0, 'tt': 1, 
    'oho': 1, '*tl': 0, '*pé': 0, '*fá': 0, '*sté': 1, '*stá': 1, 'rtr': 1, 'bp': 1, '*ví': 0, '*bse': 1, 'prei': 3, '*gü': 0, 
    '*ré': 0, 'rj': 1, '*dé': 0, 'dm': 1, '*stó': 1, 'zc': 1, 'bt': 1, 'bm': 1, 'xm': 1, 'tb': 1, '*cú': 0, '*vu': 0, 
    '*ki': 0, 'tm': 1, '*zi': 0, '*bó': 0, 'stm': 2, 'sobrei': 5, 'lz': 1, 'lq': 1, '*bso': 1, 'stp': 2, 'rh': 1, 'rk': 1, 
    'inex': 2, 'dv': 1, 'zg': 1, '*jó': 0, 'aho': 1, 'mm': 1, 'antio': 4, '*ho': 0, 'zq': 1, 'zm': 1, 'bv': 1, '*rsu': 1, 
    '*gá': 0, 'zn': 1, '*stí': 1, '*lú': 0, '*bá': 0, 'inoc': 2, '*bé': 0, 'oí': 1, '*zz': 0, '*rú': 0, 'oha': 1, 'nk': 1, 
    '*ka': 0, 'aha': 1, 'deshi': 3, '*dá': 0, '*bí': 0, 'xg': 1, 'sts': 2, 'dj': 1, 'croin': 3, 'xb': 1, '*sé': 0, '*bsi': 1, 
    '*yi': 0, '*pú': 0, 'inop': 2, '*ñu': 0, '*sá': 0, '*ke': 0, 'std': 2, 'ehe': 1, 'np': 1, 'inap': 2, 'contrai': 6, '*jí': 0, 
    'eha': 1, '*xó': 0, '*wa': 0, '*tú': 0, '*he': 0, 'sk': 1, 'ahe': 1, '*nye': 0, '*mú': 0, '*ha': 0, '*ñó': 0, '*bú': 0, 
    'pseudoi': 6, 'dh': 1, 'stg': 2, 'ln': 1, '*nú': 0, '*fta': 1, 'úa': 1, '*fé': 0, 'rw': 1, 'bb': 1, '*ze': 0, '*bsu': 1, 
    'toins': 2, 'antihi': 4, '*ku': 0, 'prehi': 3, 'ehí': 1, 'desho': 3, 'agroe': 4, 'hipn': 3, 'cd': 1, 'xcr': 1, 'ultrai': 5, 'lcr': 1, 
    'sj': 1, 'reins': 2, 'lk': 1, 'inob': 2, 'eho': 1, 'desha': 3, 'antiu': 4, 'antihe': 4, 'agroa': 4, '*ñé': 0, '*xí': 0, '*vé': 0, 
    '*vá': 0, '*já': 0, 'semiau': 4, 'rsp': 2, 'mt': 1, 'inof': 2, 'contrao': 6, 'aú': 1, '*ko': 0, 'xn': 1, 'rst': 2, 'nb': 1, 
    'hipoa': 4, 'gd': 1, 'btr': 1, '*rsá': 1, 'ms': 1, 'deshu': 3, 'antihu': 4, '*vó': 0, '*ñí': 0, '*zá': 0, '*yn': 1, '*xé': 0, 
    '*sú': 0, 'zt': 1, 'troin': 3, 'intrai': 5, 'extrai': 5, 'dc': 1, 'antiho': 4, '*yó': 0, '*yé': 0, '*xá': 0, '*jé': 0, 'yw': 1, 
    'ohí': 1, 'eú': 1, 'deshe': 3, 'cs': 1, '*nsó': 1, '*hó': 0, '*hé': 0, '*bsa': 1, 'sobreu': 5, 'ml': 1, 'mf': 1, 'mc': 1, 
    'bcr': 1, 'agroi': 4, '*yí': 0, '*yá': 0, '*nyu': 0, 'úo': 1, 'suprai': 5, 'dl': 1, 'antiha': 4, '*yú': 0, '*nsí': 1, '*hí': 0, 
    '*fú': 0, '*dú': 0, 'pseudohi': 6, 'lr': 1, 'intrau': 5, 'hipoe': 4, 'extrau': 5, 'coins': 2, 'cht': 2, 'ahú': 1, '*yc': 1, '*rsó': 1, 
    '*há': 0, '*fto': 1, 'xfr': 1, 'suprahi': 5, 'pseudoo': 6, 'preu': 3, 'miabs': 2, 'infrau': 5, 'hipoi': 4, 'dg': 1, 'ahí': 1, 'afroi': 4, 
    '*zí': 0, '*yl': 1, '*we': 0, '*vl': 0, '*stú': 1, '*nya': 0, '*fti': 1, '*fte': 1, 'úe': 1, 'ultrahu': 5, 'semiai': 4, 'rrou': 3, 
    'ohú': 1, 'kn': 1, 'intrahi': 5, 'croim': 3, 'cg': 1, 'bx': 1, 'afrohi': 4, '*yp': 1, '*ym': 1, '*wi': 0, '*rsí': 1, '*rsé': 1, 
    '*gú': 0, 'zl': 1, 'tll': 1, 'suprau': 5, 'sobrehi': 5, 'mg': 1, 'md': 1, 'infrai': 5, 'infrahu': 5, 'infrahi': 5, 'hipoo': 4, 'hipohi': 4, 
    'contrau': 6, 'ciour': 3, 'agrou': 4, 'afroo': 4, '*ñá': 0, '*zú': 0, '*wé': 0, '*vr': 0, '*pü': 0, '*nsú': 1, '*nsé': 1, '*nsá': 1, 
    '*ká': 0, '*jú': 0, '*fté': 1, '*ftá': 1, 'úho': 1, 'íe': 1, 'zv': 1, 'zr': 1, 'zj': 1, 'ultrau': 5, 'ultrahi': 5, 'suprahu': 5, 
    'sobrehu': 5, 'shb': 2, 'rñ': 1, 'pseudoho': 6, 'prehu': 3, 'oú': 1, 'mtr': 1, 'mh': 1, 'lft': 1, 'hipohu': 4, 'hipoha': 4, 'ftr': 1, 
    'extrahu': 5, 'extrahi': 5, 'ehú': 1, 'dtr': 1, 'dk': 1, 'deshá': 3, 'contrahu': 6, 'contrahi': 6, 'cohip': 2, 'antei': 4, 'antehu': 4, 'afroho': 4, 
    '*ñú': 0, '*wá': 0, '*wo': 0, '*qa': 0, '*nyo': 0, '*nyi': 0, '*kú': 0, '*kí': 0, '*hú': 0, '*ftó': 1, '*ftí': 1, '*bsó': 1, 
    '*bsí': 1, '*bsé': 1, 
    # Divisories de silaba teoricos (Sin optimizar o que no han aparecido aun en las pruebas):
    'intrahu': 5, 'semiahi': 4, 'semiahu': 4, 'anteu': 4, 'antehi': 4, 'agroo': 4, 'hipou': 4, 'hipohe': 4, 'hipoho': 4, 'deshé': 3, 
    'deshí': 3, 'deshó': 3, 'deshú': 3, 'chr': 2, 'chd': 2, 'chf': 2, 'chp': 2, 'gtr': 1, 'vtr': 1, 'jtr': 1, 'qtr': 1, 'ktr': 1, 
    'wtr': 1, 'ytr': 1, 'zcr': 1, 'ccr': 1, 'pcr': 1, 'gcr': 1, 'fcr': 1, 'íha': 1, 'úha': 1, 'íhe': 1, 'úhe': 1, 'ího': 1, 
    'nñ': 1, 'mz': 1, 'mñ': 1, 'lñ': 1, 'sx': 1, 'jl': 1, '*yr': 1, '*yd': 1, '*yf': 1, '*kl': 0, '*bü': 0, '*cü': 0, 
    '*dü': 0, '*fü': 0, '*hü': 0, '*jü': 0, '*ké': 0, '*kó': 0, '*kü': 0, '*lü': 0, '*mü': 0, '*nü': 0, '*ñü': 0, '*qe': 0, 
    '*qi': 0, '*qo': 0, '*qá': 0, '*qé': 0, '*qí': 0, '*qó': 0, '*qú': 0, '*qü': 0, '*rü': 0, '*sü': 0, '*tü': 0, '*vú': 0, 
    '*vü': 0, '*wu': 0, '*wí': 0, '*wó': 0, '*wú': 0, '*wü': 0, '*xú': 0, '*xü': 0, '*yü': 0, '*zé': 0, '*zü': 0, 'reimp': 2, 
    'brein': 3, 'troim': 3, '*bsá': 1, '*bsú': 1, '*rsú': 1, '*ftu': 1, '*ftú': 1, 
}

In [ ]:
for ttkn in dic_ngramas:
    for ngrama in dic_ngramas[ttkn]:
        if ngrama == lema:
            print(f"Para {lema}, ttkn={ttkn} tenemos {dic_ngramas[ttkn][lema]} apariciones")

In [ ]:
max_ograma = ""
max_num_ngramas = 0
for ograma in dic_contenidos:
    if len(dic_contenidos[ograma]) > max_num_ngramas:
        max_num_ngramas = len(dic_contenidos[ograma])
        max_ograma = ograma
print(f"El ngrama '{max_ograma}' aparece en {max_num_ngramas} nagramas")

In [ ]:
lista_ogramas = sorted(dic_contenidos, key=lambda x: len(dic_contenidos[x]), reverse=True)
print(len(lista_ogramas), lista_ogramas[:30])

In [ ]:
dic_continentes = {}
for ograma in lista_ogramas:
    for ngrama in dic_contenidos[ograma]:
        if ngrama in dic_continentes:
            if ograma not in dic_continentes[ngrama]:
                dic_continentes[ngrama].append(ograma)
        else:
            dic_continentes[ngrama] = [ograma]
cont = 0
for ngrama in dic_continentes:
    print(ngrama, sorted(dic_continentes[ngrama], key=lambda x: ngrama.index(x)))
    cont += 1
    if cont > 10:
        break

In [ ]:
dic_restos = {}
for ngrama in dic_continentes:
    restos = "" + ngrama
    for ograma in sorted(dic_continentes[ngrama], key=lambda x: ngrama.index(x)):
        restos = restos.replace(ograma, " ")
    for resto in restos.split():
        if resto > "":
            if resto in dic_restos:
                dic_restos[resto] += dic_ngramas[1][ngrama]
            else:
                dic_restos[resto] = dic_ngramas[1][ngrama]
print(len(dic_restos), sorted(dic_restos, key=lambda x: str(len(x)).zfill(8) + str(dic_restos[x]).zfill(11), reverse=True)[:30])

In [ ]:
fragmentos = {}
for ograma in (
    sorted(dic_continentes, key=lambda x: str(len(x)).zfill(11)+str(len(dic_continentes[x])).zfill(11)+x, reverse=True) +
    sorted(dic_restos, key=lambda x: str(len(x)).zfill(8) + str(dic_restos[x]).zfill(11), reverse=True) +
    sorted(dic_contenidos, key=lambda x: str(len(x)).zfill(11)+str(len(dic_contenidos[x])).zfill(11)+x, reverse=True)
):
    for len_parte in range(3, 7):
        for i in range(len(ograma) - len_parte):
            fragmento = ograma[i:i+len_parte]
            if fragmento in fragmentos:
                fragmentos[fragmento] += 1
            else:
                fragmentos[fragmento] = 1
print(len(fragmentos))
cont = 0
for fragmento in sorted(fragmentos, key=lambda x: fragmentos[x] + 2**len(x), reverse=True):
    print(fragmento, fragmentos[fragmento])
    cont += 1
    if cont > 20:
        break

In [ ]:
dic_monogramas = {}
tot_monogramas = 0
tot_fragmentos = 0
for ngrama in fragmentos:
    tot_fragmentos += fragmentos[ngrama]
    for caracter in ngrama:
        tot_monogramas += 1
        if caracter in dic_monogramas:
            dic_monogramas[caracter] += 1
        else:
            dic_monogramas[caracter] = 1
print(f"Con un total de apariciones de {tot_monogramas} monogramas y {tot_fragmentos} fragmentos")
print(f"Tenemos un total de  {len(dic_monogramas)} mongramas en fragmentos")

In [ ]:
fragmento_mono = {}
fragmento_n = {}
max_frag_mono_ngrama = ""
max_frag_mono = 0
max_frag_n_ngrama = ""
max_frag_n = 0
for fragmento in fragmentos:
    fragmento_n[fragmento] = fragmentos[fragmento] / tot_fragmentos
    fragmento_mono[fragmento] = 1.0
    for caracter in fragmento:
        fragmento_mono[fragmento] *= dic_monogramas[caracter] / tot_monogramas
    #fragmento_mono[fragmento] = fragmento_mono[fragmento] / len(fragmento)
    if fragmento_mono[fragmento] > max_frag_mono:
        max_frag_mono = fragmento_mono[fragmento]
        max_frag_mono_ngrama = fragmento
    if fragmento_n[fragmento] > max_frag_n:
        max_frag_n = fragmento_n[fragmento]
        max_frag_n_ngrama = fragmento
print(f"Máximo fragmento mono: '{max_frag_mono_ngrama}' = {round(max_frag_mono, 7)}")
print(f"Máximo fragmento N: '{max_frag_n_ngrama}' = {round(max_frag_n, 7)}")
suma_pab = 0.0
for fragmento in fragmentos:
    suma_pab += fragmento_n[fragmento] * fragmento_mono[fragmento]
print(f"La suma condicionada es: {suma_pab}")
cont = 0
for fragmento in sorted(fragmentos, key=lambda x: (len(set(x))) / (1.0 + abs((fragmento_n[x]/max_frag_n) - (fragmento_mono[x] / max_frag_mono))), reverse=True):
    print(
        f"<|{fragmento}|> : {round(fragmento_n[fragmento], 7)} vs {round(fragmento_mono[fragmento], 7)} veces y relación mono-n = {round(
            (len(set(fragmento))) / (1.0 + abs((fragmento_n[fragmento]/max_frag_n) - (fragmento_mono[fragmento] / max_frag_mono))), 7
            )}".ljust(70), end="")
    cont += 1
    if cont % 3 == 0:
        print()
    if cont > 60:
        print()
        break

In [ ]:
pre_tokens = {}
#for ograma in sorted(dic_contenidos, key=lambda x: str(len(x)).zfill(11)+str(len(dic_contenidos[x])).zfill(11)+x, reverse=True):
#    pre_tokens[ograma] = 0
# Cambiamos los ogramas por sus fragmentos
for ngrama in sorted(fragmentos, key=lambda x: (len(set(x))) / (1.0 + abs((fragmento_n[x]/max_frag_n) - (fragmento_mono[x] / max_frag_mono))), reverse=True):
    if ngrama not in pre_tokens:
        pre_tokens[ngrama] = 0
# El final son caracteres sueltos
for ngrama in dic_ngramas[1]:
    for caracter in ngrama:
        if caracter not in pre_tokens:
            pre_tokens[caracter] = 0
for ttkn in dic_ngramas:
    if ttkn != 1:
        for ngrama in sorted(dic_ngramas[ttkn], key=lambda x: str(len(x)).zfill(11)+str(dic_ngramas[ttkn][x]).zfill(11)+x):
            for caracter in ngrama:
                if caracter not in pre_tokens:
                    pre_tokens[caracter] = 0
print(len(pre_tokens))
cont = 0
for token in pre_tokens:
    print(token)
    cont += 1
    if cont > 20:
        break

In [ ]:
fragmentos_evaluados = [x for x in pre_tokens]
dic_usados = {}
dic_excedentes = {}
cont_excedentes = 0
for ttkn in dic_ngramas:
    print(f"Tipo {ttkn}: {K.TTKN_DESC[ttkn]}... {len(dic_ngramas[ttkn])} n-gramas")
    print("".join(["_"]*150))
    cada = len(dic_ngramas[ttkn]) // 150
    cont = 0
    for ngrama in dic_ngramas[ttkn]:
        cont_excedentes += invent.explota_fragmento(n_grama, dic_usados, dic_excedentes, fragmentos_evaluados)
        cont += 1
        if cont % cada == 0:
            print("|", end="")
    print()
print(f"Tenemos {cont_excedentes} excedentes acumulados de {len(cont_excedentes)} excedentes distintos")
print(f"Confirmados {len(dic_usados)} tokens")

In [ ]:
for ttkn in dic_ngramas:
    if ttkn != 1:
        print(f"Tipo {ttkn}: {K.TTKN_DESC[ttkn]}... {len(dic_ngramas[ttkn])} n-gramas")
        print("".join(["_"]*150))
        cada = max([len(dic_ngramas[ttkn]) // 150, 1])
        cont = 0
        for ngrama in dic_ngramas[ttkn]:
            cont_excedentes += invent.explota_fragmento(n_grama, dic_usados, dic_excedentes, fragmentos_evaluados)
            cont += 1
            if cont % cada == 0:
                print("|", end="")
        print()
print(f"Tenemos {cont_excedentes} excedentes acumulados de {len(dic_excedentes)} excedentes distintos")
print(f"Confirmados {len(dic_usados)} tokens")

In [ ]:
print(f"Tenemos {cont_excedentes} excedentes acumulados de {len(dic_excedentes)} excedentes distintos")
print(f"Confirmados {len(dic_usados)} tokens")

In [ ]:
print(dic_usados)

```
lo administrativo es muy atractivo
l-o- -a-d-m-i-n-i-s-t-r-a-t-i-v-o- -e-s- -m-u-y- -a-t-r-a-c-t-i-v-o = 34
l=1/34 o=4/34  =4/34 a=3/34 d=1/34 m=2/34 i=4/34 n=1/34 s=2/34 t=4/34 r=2/34 v=2/34 e=1/34 u=1/34 y=1/34 c=1/34

lo- a-dm-in-is-tr-at-iv-o -es- m-uy- a-tr-ac-ti-vo-o -ad-mi-ni-st-ra-ti-vo- e-s -mu-y -at-ra-ct-iv
lo=1/34 ' a'=2/34 dm=1/34 in=1/34 is=1/34 tr=2/34 at=2/34 iv=2/34 'o '=2/34 es=1/34 ' m'=1/34 uy=1/34 ac=1/34 ti=2/34 vo=2/34 ad=2/34 mi=1/34 ni=1/34 st=1/34 ra=2/34 ' e'=1/34 's '=1/34 mu=1/34 'y '=1/34 ct=1/34

lo -adm-ini-str-ati-vo -es -muy- at-rac-tiv-o a-dmi-nis-tra-tiv-o e-s m-uy -atr-act-ivo- ad-min-ist-rat-ivo- es- mu-y a-tra-cti
'lo '=1/32 adm=1/32 ini=1/32 str=1/32 ati=1/32 'vo '=1/32 'es '=1/32 muy=1/32 ' at'=1/32 rac=1/32 tiv=2/32 'o a'=1/32 dmi=1/32 nis=1/32 tra=2/32 'o e'=1/32 's m'=1/32 'uy '=1/32 atr=1/32 act=1/32 ivo=2/32 ' ad'=1/32 min=1/32 ist=1/32 rat=1/32 ' es'=1/32 ' mu'=1/32 'y a'=1/32 cti=1/32

ORDEN:
 =4/34  =0.117647

a=3/34  =0.0882 
+tiv=2/32=0.0625
/t=4/34  =0.117647
+tra=2/32
ivo=2/32
' a'=2/34
+tr=2/34
-at=2/34
iv=2/34
/i=4/34  =0.117647
-'o '=2/34
-ti=2/34
+vo=2/34
/o=4/34  =0.117647
ad=2/34
ra=2/34
m=2/34
s=2/34
r=2/34
v=2/34
'lo '=1/32
adm=1/32
+ini=1/32
-str=1/32
-ati=1/32
'vo '=1/32
'es '=1/32
muy=1/32
' at'=1/32
rac=1/32
'o a'=1/32
dmi=1/32
nis=1/32
'o e'=1/32
's m'=1/32
'uy '=1/32
atr=1/32
act=1/32
' ad'=1/32
min=1/32
ist=1/32
rat=1/32
' es'=1/32
' mu'=1/32
'y a'=1/32
cti=1/32
lo=1/34
dm=1/34
in=1/34
is=1/34
es=1/34
' m'=1/34
uy=1/34
ac=1/34
mi=1/34
ni=1/34
st=1/34
' e'=1/34
's '=1/34
mu=1/34
'y '=1/34
ct=1/34
l=1/34
d=1/34
n=1/34
e=1/34
u=1/34
y=1/34
c=1/34



lo a-dmin-istr-ativ-o es- muy- atr-acti--o ad-mini-stra-tivo- es -muy -atra-ctiv-- adm-inis-trat-ivo -es m-uy a-trac-tivo-admi-nist-rati-vo e-s mu-y at-ract-

lo ad-minis-trati-vo es- muy -atrac--o adm-inist-rativ-o es -muy a-tract-- admi-nistr-ativo- es m-uy at-racti--admin-istra-tivo -es mu-y atr-activ--dmini-strat-ivo e-s muy- atra-ctivo

lo adm-inistr-ativo -es muy- atrac--o admi-nistra-tivo e-s muy -atract-- admin-istrat-ivo es- muy a-tracti--admini-strati-vo es -muy at-ractiv--dminis-trativ-o es m-uy atr-activo-minist-rativo- es mu-y atra-

34 monogramas
34 bigramas




```